# FileVersion/IPEDS_ALL_ORIGINAL_INFO_FIXED_V4.csv

In [4]:
from pathlib import Path
import time

files = sorted(
    Path(".").glob("FileVersion/IPEDS_ALL_ORIGINAL_INFO_FIXED_V*.csv"),
    key=lambda x: x.stat().st_mtime,
    reverse=True
)

print("Matching files:")
print("=" * 60)

for f in files:
    modified = time.strftime("%Y-%m-%d %H:%M:%S", time.localtime(f.stat().st_mtime))
    size_mb = f.stat().st_size / (1024 * 1024)
    print(f"{modified} | {size_mb:,.2f} MB | {f.name}")

if files:
    print("\nNewest version appears to be:")
    print(files[0].name)
else:
    print("No V files found.")

Matching files:
2026-06-09 11:29:07 | 3,799.96 MB | IPEDS_ALL_ORIGINAL_INFO_FIXED_V4.csv
2026-06-09 03:37:30 | 3,783.54 MB | IPEDS_ALL_ORIGINAL_INFO_FIXED_V3.csv

Newest version appears to be:
IPEDS_ALL_ORIGINAL_INFO_FIXED_V4.csv


In [10]:
import pandas as pd
from pathlib import Path
from collections import Counter, defaultdict
import time
import math
import gc

# ==================================================
# PROJECT FOLDER
# ==================================================

folder = Path("FileVersion")

if not folder.exists():
    raise FileNotFoundError(f"Folder not found: {folder}")

# Automatically find newest fixed version file inside FileVersion
files = sorted(
    folder.glob("IPEDS_ALL_ORIGINAL_INFO_FIXED_V*.csv"),
    key=lambda x: x.stat().st_mtime,
    reverse=True
)

if not files:
    print("No IPEDS_ALL_ORIGINAL_INFO_FIXED_V*.csv found inside FileVersion.")
    print("\nFiles inside FileVersion:")
    for f in folder.glob("*"):
        print(" -", f.name)
    raise FileNotFoundError("No fixed version CSV found.")

file = files[0]

# ==================================================
# SETTINGS
# ==================================================

CHUNKSIZE = 100_000
PRINT_EVERY_CHUNKS = 5
COUNT_ROWS_FIRST = True

# True = check missing values in every column
# False = only check important project columns, faster
CHECK_ALL_COLUMNS_FOR_MISSING = True

MAX_PROBLEM_SAMPLE_ROWS = 2000

# Report names based on newest file name
base_name = file.stem

REPORT_FILE = folder / f"{base_name}_PROJECT_CHECK_REPORT.csv"
PROBLEM_SAMPLE_FILE = folder / f"{base_name}_PROBLEM_SAMPLE_ROWS.csv"
YEAR_COUNT_FILE = folder / f"{base_name}_YEAR_COUNTS.csv"
FIELD_GROUP_COUNT_FILE = folder / f"{base_name}_FIELD_GROUP_COUNTS.csv"
DEGREE_GROUP_COUNT_FILE = folder / f"{base_name}_DEGREE_GROUP_COUNTS.csv"

# ==================================================
# BASIC FILE INFO
# ==================================================

file_size_mb = file.stat().st_size / (1024 * 1024)

print("Using newest project file:")
print(file)
print(f"File size: {file_size_mb:,.2f} MB")

# ==================================================
# HELPER FUNCTIONS
# ==================================================

def fmt_time(seconds):
    if seconds is None or seconds <= 0 or math.isnan(seconds) or math.isinf(seconds):
        return "calculating..."

    seconds = int(seconds)
    h = seconds // 3600
    m = (seconds % 3600) // 60
    s = seconds % 60

    if h > 0:
        return f"{h}h {m}m {s}s"
    if m > 0:
        return f"{m}m {s}s"
    return f"{s}s"


def count_csv_rows_fast(path):
    """
    Count CSV rows without loading the file into memory.
    Used only for progress and ETA.
    """
    total_bytes = path.stat().st_size
    bytes_read = 0
    line_breaks = 0
    last_byte = b""

    start = time.time()
    last_print = 0
    block_size = 32 * 1024 * 1024

    with open(path, "rb") as f:
        while True:
            block = f.read(block_size)

            if not block:
                break

            bytes_read += len(block)
            line_breaks += block.count(b"\n")
            last_byte = block[-1:]

            now = time.time()

            if now - last_print >= 3:
                pct = bytes_read / total_bytes
                elapsed = now - start
                eta = (elapsed / pct) - elapsed if pct > 0 else None

                print(
                    f"Counting rows for ETA: {pct:6.2%} | "
                    f"Elapsed: {fmt_time(elapsed)} | ETA: {fmt_time(eta)}",
                    end="\r"
                )

                last_print = now

    total_lines = line_breaks

    if last_byte not in (b"\n", b"\r"):
        total_lines += 1

    data_rows = max(total_lines - 1, 0)

    print()
    return data_rows


def is_missing_series(s):
    """
    Treat blank, nan, none, null, and <na> as missing.
    """
    ss = s.astype(str).str.strip()

    return (
        ss.eq("")
        | ss.str.lower().isin(["nan", "none", "null", "<na>"])
    )


def find_col(columns, candidates):
    """
    Finds a column even if uppercase/lowercase is different.
    """
    lower_map = {c.lower().strip(): c for c in columns}

    for name in candidates:
        key = name.lower().strip()

        if key in lower_map:
            return lower_map[key]

    return None


def save_counter(counter, filename, name_col, count_col):
    df = pd.DataFrame(counter.items(), columns=[name_col, count_col])

    if not df.empty:
        df = df.sort_values(count_col, ascending=False)

    df.to_csv(filename, index=False)
    print(f"Saved: {filename}")


def show_df(df, n=30):
    try:
        display(df.head(n))
    except NameError:
        print(df.head(n))


# ==================================================
# READ HEADER ONLY
# ==================================================

header = pd.read_csv(file, nrows=0, low_memory=False)
columns = header.columns.tolist()

print("\nColumns found:")
for c in columns:
    print(" -", c)

# ==================================================
# FIND IMPORTANT PROJECT COLUMNS
# ==================================================

found = {}

found["year"] = find_col(columns, ["year", "YEAR"])
found["unitid"] = find_col(columns, ["unitid", "UNITID", "institution_id"])
found["cipcode"] = find_col(columns, ["cipcode", "cip_code", "CIPCODE", "cip", "CIP"])
found["award_level"] = find_col(columns, ["award_level", "award_level_code", "AWLEVEL", "awlevel"])
found["degree_level"] = find_col(columns, ["degree_level"])
found["degree_group"] = find_col(columns, ["degree_group"])
found["major_name"] = find_col(columns, ["major_name", "clean_major_name", "cip_title", "program_title"])
found["field_group"] = find_col(columns, ["field_group"])
found["field_subgroup"] = find_col(columns, ["field_subgroup"])
found["degree_count"] = find_col(columns, ["degree_count", "count", "total", "CTOTALT", "ctotalt"])

print("\nImportant project columns found:")
for k, v in found.items():
    print(f"{k:15} -> {v}")

missing_project_columns = [k for k, v in found.items() if v is None]

if missing_project_columns:
    print("\nWARNING: These project columns were not found:")
    for c in missing_project_columns:
        print(" -", c)
else:
    print("\nAll main project columns were found.")

# ==================================================
# SMALL PREVIEW LOAD ONLY
# ==================================================

print("\nLoading preview only...")

preview = pd.read_csv(
    file,
    nrows=10,
    dtype=str,
    keep_default_na=False,
    low_memory=False
)

print("\nPreview:")
show_df(preview, 10)

# ==================================================
# COUNT TOTAL ROWS FIRST FOR ETA
# ==================================================

expected_rows = None

if COUNT_ROWS_FIRST:
    print("\nCounting total rows first for loading progress...")
    expected_rows = count_csv_rows_fast(file)
    print(f"Expected total data rows: {expected_rows:,}")

# ==================================================
# CHOOSE COLUMNS TO READ
# ==================================================

project_cols = [v for v in found.values() if v is not None]
project_cols = list(dict.fromkeys(project_cols))

if CHECK_ALL_COLUMNS_FOR_MISSING:
    usecols = None
    print("\nMode: checking missing values in ALL columns.")
else:
    usecols = project_cols
    print("\nMode: checking only important project columns.")
    print("Columns used:", usecols)

# ==================================================
# MEMORY-SAFE FULL PROJECT CHECK
# ==================================================

total_rows = 0
start_time = time.time()

missing_counts = defaultdict(int)
unknown_counts = defaultdict(int)
unmapped_counts = defaultdict(int)

year_counts = Counter()
field_group_counts = Counter()
field_subgroup_counts = Counter()
degree_group_counts = Counter()

bad_year_rows = 0
bad_degree_count_rows = 0
negative_degree_count_rows = 0
degree_count_total = 0

problem_parts = []
problem_sample_count = 0

expected_year_start = 1984
expected_year_end = 2024

print("\nStarting memory-safe project check...\n")

reader = pd.read_csv(
    file,
    chunksize=CHUNKSIZE,
    dtype=str,
    keep_default_na=False,
    usecols=usecols,
    low_memory=False
)

for chunk_number, chunk in enumerate(reader, start=1):

    total_rows += len(chunk)

    # ----------------------------------------------
    # Missing check
    # ----------------------------------------------
    for col in chunk.columns:
        missing_counts[col] += int(is_missing_series(chunk[col]).sum())

    # ----------------------------------------------
    # Year check
    # ----------------------------------------------
    year_col = found.get("year")

    if year_col in chunk.columns:
        raw_year = chunk[year_col].astype(str).str.strip()
        year_num = pd.to_numeric(raw_year, errors="coerce")

        bad_year_mask = year_num.isna() & raw_year.ne("")
        bad_year_rows += int(bad_year_mask.sum())

        valid_years = year_num.dropna().astype(int)
        year_counts.update(valid_years.value_counts().to_dict())

    # ----------------------------------------------
    # Degree count check
    # ----------------------------------------------
    count_col = found.get("degree_count")

    if count_col in chunk.columns:
        raw_count = (
            chunk[count_col]
            .astype(str)
            .str.replace(",", "", regex=False)
            .str.strip()
        )

        count_num = pd.to_numeric(raw_count, errors="coerce")

        bad_count_mask = count_num.isna() & raw_count.ne("")
        bad_degree_count_rows += int(bad_count_mask.sum())

        negative_mask = count_num < 0
        negative_degree_count_rows += int(negative_mask.sum())

        degree_count_total += count_num.fillna(0).sum()

    # ----------------------------------------------
    # Unknown / Unmapped label check
    # ----------------------------------------------
    label_cols = [
        found.get("field_group"),
        found.get("field_subgroup"),
        found.get("major_name"),
        found.get("degree_group"),
        found.get("degree_level"),
    ]

    label_cols = [c for c in label_cols if c in chunk.columns]

    for col in label_cols:
        s = chunk[col].astype(str).str.strip().str.lower()

        unknown_mask = s.str.startswith("unknown")
        unmapped_mask = s.str.contains("unmapped", regex=False)

        unknown_counts[col] += int(unknown_mask.sum())
        unmapped_counts[col] += int(unmapped_mask.sum())

    # ----------------------------------------------
    # Useful project group counts
    # ----------------------------------------------
    fg_col = found.get("field_group")

    if fg_col in chunk.columns:
        s = chunk[fg_col].astype(str).str.strip()
        s = s[s.ne("")]
        field_group_counts.update(s.value_counts().to_dict())

    fsg_col = found.get("field_subgroup")

    if fsg_col in chunk.columns:
        s = chunk[fsg_col].astype(str).str.strip()
        s = s[s.ne("")]
        field_subgroup_counts.update(s.value_counts().to_dict())

    dg_col = found.get("degree_group")

    if dg_col in chunk.columns:
        s = chunk[dg_col].astype(str).str.strip()
        s = s[s.ne("")]
        degree_group_counts.update(s.value_counts().to_dict())

    elif found.get("degree_level") in chunk.columns:
        dl_col = found.get("degree_level")
        s = chunk[dl_col].astype(str).str.strip()
        s = s[s.ne("")]
        degree_group_counts.update(s.value_counts().to_dict())

    # ----------------------------------------------
    # Save small sample of problem rows
    # ----------------------------------------------
    if problem_sample_count < MAX_PROBLEM_SAMPLE_ROWS:
        problem_mask = pd.Series(False, index=chunk.index)

        important_for_problem = [
            found.get("year"),
            found.get("cipcode"),
            found.get("major_name"),
            found.get("field_group"),
            found.get("field_subgroup"),
            found.get("degree_count"),
        ]

        important_for_problem = [c for c in important_for_problem if c in chunk.columns]

        for col in important_for_problem:
            problem_mask = problem_mask | is_missing_series(chunk[col])

        for col in label_cols:
            s = chunk[col].astype(str).str.strip().str.lower()

            problem_mask = (
                problem_mask
                | s.str.startswith("unknown")
                | s.str.contains("unmapped", regex=False)
            )

        if problem_mask.any():
            remaining = MAX_PROBLEM_SAMPLE_ROWS - problem_sample_count
            sample = chunk.loc[problem_mask].head(remaining).copy()
            sample.insert(0, "source_chunk", chunk_number)

            problem_parts.append(sample)
            problem_sample_count += len(sample)

    # ----------------------------------------------
    # Progress loading
    # ----------------------------------------------
    if chunk_number % PRINT_EVERY_CHUNKS == 0:
        elapsed = time.time() - start_time

        if expected_rows:
            pct = total_rows / expected_rows
            eta = (elapsed / pct) - elapsed if pct > 0 else None

            print(
                f"Loaded/checked: {total_rows:,} / {expected_rows:,} rows "
                f"({pct:6.2%}) | Chunk {chunk_number:,} | "
                f"Elapsed: {fmt_time(elapsed)} | ETA: {fmt_time(eta)}"
            )

        else:
            print(
                f"Loaded/checked: {total_rows:,} rows | "
                f"Chunk {chunk_number:,} | Elapsed: {fmt_time(elapsed)}"
            )

    del chunk
    gc.collect()

# ==================================================
# BUILD REPORT
# ==================================================

print("\nBuilding report...")

report_rows = []

report_rows.append({
    "check_type": "file",
    "column": "",
    "bad_rows": "",
    "percent_of_rows": "",
    "note": f"File checked: {file}; size: {file_size_mb:,.2f} MB"
})

report_rows.append({
    "check_type": "total_rows",
    "column": "",
    "bad_rows": total_rows,
    "percent_of_rows": "",
    "note": "Total rows checked"
})

for logical_name, actual_col in found.items():
    if actual_col is None:
        report_rows.append({
            "check_type": "missing_project_column",
            "column": logical_name,
            "bad_rows": "",
            "percent_of_rows": "",
            "note": "Column was not found in file"
        })

for col in columns:
    if col in missing_counts:
        bad = missing_counts[col]
        pct = (bad / total_rows * 100) if total_rows else 0

        report_rows.append({
            "check_type": "blank_or_null_values",
            "column": col,
            "bad_rows": bad,
            "percent_of_rows": round(pct, 6),
            "note": "Blank, nan, none, null, or <na>"
        })

for col, bad in unknown_counts.items():
    pct = (bad / total_rows * 100) if total_rows else 0

    report_rows.append({
        "check_type": "unknown_label_values",
        "column": col,
        "bad_rows": bad,
        "percent_of_rows": round(pct, 6),
        "note": "Values starting with Unknown"
    })

for col, bad in unmapped_counts.items():
    pct = (bad / total_rows * 100) if total_rows else 0

    report_rows.append({
        "check_type": "unmapped_label_values",
        "column": col,
        "bad_rows": bad,
        "percent_of_rows": round(pct, 6),
        "note": "Transparent nonstandard Unmapped labels"
    })

if found.get("year") is not None:
    pct = (bad_year_rows / total_rows * 100) if total_rows else 0

    report_rows.append({
        "check_type": "bad_year_values",
        "column": found.get("year"),
        "bad_rows": bad_year_rows,
        "percent_of_rows": round(pct, 6),
        "note": "Year could not be converted to number"
    })

    years_found = sorted(year_counts.keys())
    expected_years = list(range(expected_year_start, expected_year_end + 1))
    missing_years = [y for y in expected_years if y not in year_counts]

    report_rows.append({
        "check_type": "year_range_found",
        "column": found.get("year"),
        "bad_rows": "",
        "percent_of_rows": "",
        "note": f"Years found from {min(years_found) if years_found else 'None'} to {max(years_found) if years_found else 'None'}"
    })

    report_rows.append({
        "check_type": "missing_expected_years",
        "column": found.get("year"),
        "bad_rows": len(missing_years),
        "percent_of_rows": "",
        "note": f"Expected {expected_year_start}-{expected_year_end}; missing years: {missing_years}"
    })

if found.get("degree_count") is not None:
    pct_bad = (bad_degree_count_rows / total_rows * 100) if total_rows else 0
    pct_neg = (negative_degree_count_rows / total_rows * 100) if total_rows else 0

    report_rows.append({
        "check_type": "bad_degree_count_values",
        "column": found.get("degree_count"),
        "bad_rows": bad_degree_count_rows,
        "percent_of_rows": round(pct_bad, 6),
        "note": "Degree count could not be converted to number"
    })

    report_rows.append({
        "check_type": "negative_degree_count_values",
        "column": found.get("degree_count"),
        "bad_rows": negative_degree_count_rows,
        "percent_of_rows": round(pct_neg, 6),
        "note": "Degree count is negative"
    })

    report_rows.append({
        "check_type": "degree_count_total",
        "column": found.get("degree_count"),
        "bad_rows": "",
        "percent_of_rows": "",
        "note": f"Total degree count sum: {degree_count_total:,.0f}"
    })

report = pd.DataFrame(report_rows)
report.to_csv(REPORT_FILE, index=False)

print(f"Saved: {REPORT_FILE}")

# ==================================================
# SAVE PROBLEM SAMPLE ROWS
# ==================================================

if problem_parts:
    problem_sample = pd.concat(problem_parts, ignore_index=True)
    problem_sample.to_csv(PROBLEM_SAMPLE_FILE, index=False)
    print(f"Saved: {PROBLEM_SAMPLE_FILE}")
else:
    print("No problem sample rows found.")

# ==================================================
# SAVE COUNT FILES
# ==================================================

if year_counts:
    save_counter(year_counts, YEAR_COUNT_FILE, "year", "rows")

if field_group_counts:
    save_counter(field_group_counts, FIELD_GROUP_COUNT_FILE, "field_group", "rows")

if degree_group_counts:
    save_counter(degree_group_counts, DEGREE_GROUP_COUNT_FILE, "degree_group", "rows")

# Optional subgroup count file
FIELD_SUBGROUP_COUNT_FILE = folder / f"{base_name}_FIELD_SUBGROUP_COUNTS.csv"

if field_subgroup_counts:
    save_counter(field_subgroup_counts, FIELD_SUBGROUP_COUNT_FILE, "field_subgroup", "rows")

# ==================================================
# PRINT FINAL SUMMARY
# ==================================================

elapsed_total = time.time() - start_time

print("\nDONE")
print("=" * 70)
print(f"File checked: {file}")
print(f"Total rows checked: {total_rows:,}")
print(f"Total time: {fmt_time(elapsed_total)}")

if expected_rows:
    print(f"Expected rows from row count: {expected_rows:,}")

    if expected_rows == total_rows:
        print("Row count check: OK")
    else:
        print("Row count check: DIFFERENT - check file carefully")

print("\nMain project columns:")
for k, v in found.items():
    print(f"{k:15} -> {v}")

print("\nImportant problem counts:")
print(f"Bad year rows: {bad_year_rows:,}")
print(f"Bad degree_count rows: {bad_degree_count_rows:,}")
print(f"Negative degree_count rows: {negative_degree_count_rows:,}")
print(f"Total degree_count sum: {degree_count_total:,.0f}")

print("\nUnknown label rows:")
if unknown_counts:
    for col, count in unknown_counts.items():
        print(f"{col}: {count:,}")
else:
    print("None found.")

print("\nUnmapped label rows:")
if unmapped_counts:
    for col, count in unmapped_counts.items():
        print(f"{col}: {count:,}")
else:
    print("None found.")

if year_counts:
    years_found = sorted(year_counts.keys())
    missing_years = [y for y in range(expected_year_start, expected_year_end + 1) if y not in year_counts]

    print("\nYear check:")
    print(f"First year found: {min(years_found)}")
    print(f"Last year found: {max(years_found)}")
    print(f"Missing expected years {expected_year_start}-{expected_year_end}: {missing_years}")

print("\nTop report rows with problems:")
problem_report = report.copy()
problem_report["bad_rows_num"] = pd.to_numeric(problem_report["bad_rows"], errors="coerce").fillna(0)
problem_report = problem_report[problem_report["bad_rows_num"] > 0]
problem_report = problem_report.sort_values("bad_rows_num", ascending=False)

if len(problem_report) > 0:
    show_df(problem_report.drop(columns=["bad_rows_num"]), 40)
else:
    print("No problem counts above 0.")

print("\nFiles created:")
print(" -", REPORT_FILE)
print(" -", PROBLEM_SAMPLE_FILE)
print(" -", YEAR_COUNT_FILE)
print(" -", FIELD_GROUP_COUNT_FILE)
print(" -", DEGREE_GROUP_COUNT_FILE)
print(" -", FIELD_SUBGROUP_COUNT_FILE)

Using newest project file:
FileVersion/IPEDS_ALL_ORIGINAL_INFO_FIXED_V4.csv
File size: 3,799.96 MB

Columns found:
 - unitid
 - cipcode
 - awlevel
 - crace15
 - crace16
 - part
 - crace01
 - crace02
 - crace03
 - crace04
 - crace05
 - crace06
 - crace07
 - crace08
 - crace09
 - crace10
 - crace11
 - crace12
 - crace13
 - crace14
 - bal_m
 - bal_w
 - imprac01
 - imprac02
 - imprac03
 - imprac04
 - imprac05
 - imprac06
 - imprac07
 - imprac08
 - imprac09
 - imprac10
 - imprac11
 - imprac12
 - imprac13
 - imprac14
 - imprac15
 - imprac16
 - ix01
 - ix02
 - ix03
 - ix04
 - ix05
 - ix06
 - ix07
 - ix08
 - ix09
 - ix10
 - ix11
 - ix12
 - ix13
 - ix14
 - ix15
 - ix16
 - cbalm
 - cbalw
 - xcrace01
 - xcrace02
 - xcrace03
 - xcrace04
 - xcrace05
 - xcrace06
 - xcrace07
 - xcrace08
 - xcrace09
 - xcrace10
 - xcrace11
 - xcrace12
 - xcrace13
 - xcrace14
 - xcrace15
 - xcrace16
 - xcrace17
 - crace17
 - xcrace18
 - crace18
 - xcrace19
 - crace19
 - xcrace20
 - crace20
 - xcrace21
 - crace21
 - xcr

,unitid,cipcode,awlevel,crace15,crace16,part,crace01,crace02,crace03,crace04,...,CDISTEDP,year,award_level_code,award_level_name,degree_group,cip_code,degree_count,field_group,field_subgroup,major_name
0,100654,10102,5,11,0,,,,,,...,,1984,5,Bachelor,Bachelor,10102,0,COMMUNICATIONS TECHNOLOGIES/TECHNICIANS AND SU...,Unmapped / Nonstandard CIP Subgroup (10102),Unknown Major
1,100654,10102,7,14,0,,,,,,...,,1984,7,Master,Master,10102,0,COMMUNICATIONS TECHNOLOGIES/TECHNICIANS AND SU...,Unmapped / Nonstandard CIP Subgroup (10102),Unknown Major
2,100654,10103,5,1,1,,,,,,...,,1984,5,Bachelor,Bachelor,10103,0,COMMUNICATIONS TECHNOLOGIES/TECHNICIANS AND SU...,Unmapped / Nonstandard CIP Subgroup (10103),Unknown Major
3,100654,20201,5,3,0,,,,,,...,,1984,5,Bachelor,Bachelor,20201,0,Unmapped / Nonstandard CIP Field (20),Unmapped / Nonstandard CIP Subgroup (20201),Unknown Major
4,100654,20301,5,3,0,,,,,,...,,1984,5,Bachelor,Bachelor,20301,0,Unmapped / Nonstandard CIP Field (20),Unmapped / Nonstandard CIP Subgroup (20301),Unknown Major
5,100654,20399,7,11,3,,,,,,...,,1984,7,Master,Master,20399,0,Unmapped / Nonstandard CIP Field (20),Unmapped / Nonstandard CIP Subgroup (20399),Unknown Major
6,100654,20501,5,2,0,,,,,,...,,1984,5,Bachelor,Bachelor,20501,0,Unmapped / Nonstandard CIP Field (20),Unmapped / Nonstandard CIP Subgroup (20501),Unknown Major
7,100654,29999,5,1,0,,,,,,...,,1984,5,Bachelor,Bachelor,29999,0,MILITARY TECHNOLOGIES AND APPLIED SCIENCES,Unmapped / Nonstandard CIP Subgroup (29999),Unknown Major
8,100654,29999,7,10,0,,,,,,...,,1984,7,Master,Master,29999,0,MILITARY TECHNOLOGIES AND APPLIED SCIENCES,Unmapped / Nonstandard CIP Subgroup (29999),Unknown Major
9,100654,30499,5,2,0,,,,,,...,,1984,5,Bachelor,Bachelor,30499,0,MULTI/INTERDISCIPLINARY STUDIES,Unmapped / Nonstandard CIP Subgroup (30499),Unknown Major



Counting total rows first for loading progress...
Counting rows for ETA:  0.84% | Elapsed: 0s | ETA: 2s
Expected total data rows: 8,830,649

Mode: checking missing values in ALL columns.

Starting memory-safe project check...

Loaded/checked: 500,000 / 8,830,649 rows ( 5.66%) | Chunk 5 | Elapsed: 25s | ETA: 7m 7s
Loaded/checked: 1,000,000 / 8,830,649 rows (11.32%) | Chunk 10 | Elapsed: 51s | ETA: 6m 39s
Loaded/checked: 1,500,000 / 8,830,649 rows (16.99%) | Chunk 15 | Elapsed: 1m 17s | ETA: 6m 16s
Loaded/checked: 2,000,000 / 8,830,649 rows (22.65%) | Chunk 20 | Elapsed: 1m 45s | ETA: 5m 58s
Loaded/checked: 2,500,000 / 8,830,649 rows (28.31%) | Chunk 25 | Elapsed: 2m 12s | ETA: 5m 36s
Loaded/checked: 3,000,000 / 8,830,649 rows (33.97%) | Chunk 30 | Elapsed: 2m 41s | ETA: 5m 13s
Loaded/checked: 3,500,000 / 8,830,649 rows (39.63%) | Chunk 35 | Elapsed: 3m 9s | ETA: 4m 48s
Loaded/checked: 4,000,000 / 8,830,649 rows (45.30%) | Chunk 40 | Elapsed: 3m 38s | ETA: 4m 23s
Loaded/checked: 4,500,0

,check_type,column,bad_rows,percent_of_rows,note
1,total_rows,,8830649,,Total rows checked
58,blank_or_null_values,cbalw,8830649,100.0,"Blank, nan, none, null, or <na>"
57,blank_or_null_values,cbalm,8830649,100.0,"Blank, nan, none, null, or <na>"
48,blank_or_null_values,ix08,8628824,97.714494,"Blank, nan, none, null, or <na>"
47,blank_or_null_values,ix07,8628824,97.714494,"Blank, nan, none, null, or <na>"
46,blank_or_null_values,ix06,8628824,97.714494,"Blank, nan, none, null, or <na>"
49,blank_or_null_values,ix09,8628824,97.714494,"Blank, nan, none, null, or <na>"
45,blank_or_null_values,ix05,8628824,97.714494,"Blank, nan, none, null, or <na>"
44,blank_or_null_values,ix04,8628824,97.714494,"Blank, nan, none, null, or <na>"
51,blank_or_null_values,ix11,8628824,97.714494,"Blank, nan, none, null, or <na>"



Files created:
 - FileVersion/IPEDS_ALL_ORIGINAL_INFO_FIXED_V4_PROJECT_CHECK_REPORT.csv
 - FileVersion/IPEDS_ALL_ORIGINAL_INFO_FIXED_V4_PROBLEM_SAMPLE_ROWS.csv
 - FileVersion/IPEDS_ALL_ORIGINAL_INFO_FIXED_V4_YEAR_COUNTS.csv
 - FileVersion/IPEDS_ALL_ORIGINAL_INFO_FIXED_V4_FIELD_GROUP_COUNTS.csv
 - FileVersion/IPEDS_ALL_ORIGINAL_INFO_FIXED_V4_DEGREE_GROUP_COUNTS.csv
 - FileVersion/IPEDS_ALL_ORIGINAL_INFO_FIXED_V4_FIELD_SUBGROUP_COUNTS.csv


# FileVersion/IPEDS_ALL_ORIGINAL_INFO_FIXED_V4.csv

In [24]:
import pandas as pd
from pathlib import Path
import time

# ==================================================
# EXACT PROJECT FILE
# ==================================================

file = Path("FileVersion") / "IPEDS_ALL_ORIGINAL_INFO_FIXED_V4.csv"

if not file.exists():
    raise FileNotFoundError(f"File not found: {file}")

print("Using file:", file)

# ==================================================
# MEMORY OPTIMIZATION
# Only load needed columns
# ==================================================

usecols = ["field_group", "field_subgroup", "major_name"]
chunksize = 200_000

total = 0
unmapped_group = 0
unmapped_subgroup = 0
unmapped_major = 0

start = time.time()

# ==================================================
# CHECK UNMAPPED LABELS
# ==================================================

for i, chunk in enumerate(
    pd.read_csv(
        file,
        usecols=usecols,
        chunksize=chunksize,
        dtype=str,
        keep_default_na=False,
        low_memory=False
    ),
    start=1
):
    total += len(chunk)

    fg = chunk["field_group"].str.strip()
    fs = chunk["field_subgroup"].str.strip()
    mn = chunk["major_name"].str.strip()

    unmapped_group += fg.str.startswith("Unmapped Field Group").sum()
    unmapped_subgroup += fs.str.startswith("Unmapped Subgroup").sum()
    unmapped_major += mn.str.startswith("Unmapped").sum()

    print(
        f"Loaded chunk {i} | Rows checked: {total:,} | "
        f"Time: {time.time() - start:,.1f} sec"
    )

# ==================================================
# FINAL OUTPUT
# ==================================================

print("\nDONE")
print(f"Total rows checked: {total:,}")

print("\nTransparent nonstandard labels:")
print(f"Unmapped field group rows: {unmapped_group:,}")
print(f"Unmapped subgroup rows: {unmapped_subgroup:,}")
print(f"Unmapped major rows: {unmapped_major:,}")

print(f"\nTotal time: {time.time() - start:,.1f} seconds")

Using file: FileVersion/IPEDS_ALL_ORIGINAL_INFO_FIXED_V4.csv
Loaded chunk 1 | Rows checked: 200,000 | Time: 0.6 sec
Loaded chunk 2 | Rows checked: 400,000 | Time: 1.2 sec
Loaded chunk 3 | Rows checked: 600,000 | Time: 1.8 sec
Loaded chunk 4 | Rows checked: 800,000 | Time: 2.4 sec
Loaded chunk 5 | Rows checked: 1,000,000 | Time: 3.0 sec
Loaded chunk 6 | Rows checked: 1,200,000 | Time: 3.5 sec
Loaded chunk 7 | Rows checked: 1,400,000 | Time: 4.1 sec
Loaded chunk 8 | Rows checked: 1,600,000 | Time: 4.7 sec
Loaded chunk 9 | Rows checked: 1,800,000 | Time: 5.4 sec
Loaded chunk 10 | Rows checked: 2,000,000 | Time: 6.0 sec
Loaded chunk 11 | Rows checked: 2,200,000 | Time: 6.6 sec
Loaded chunk 12 | Rows checked: 2,400,000 | Time: 7.3 sec
Loaded chunk 13 | Rows checked: 2,600,000 | Time: 7.9 sec
Loaded chunk 14 | Rows checked: 2,800,000 | Time: 8.7 sec
Loaded chunk 15 | Rows checked: 3,000,000 | Time: 9.3 sec
Loaded chunk 16 | Rows checked: 3,200,000 | Time: 10.0 sec
Loaded chunk 17 | Rows chec

# Check summary

In [30]:
import pandas as pd
from pathlib import Path

# ==================================================
# FILE
# ==================================================

file = Path("FileVersion/IPEDS_ALL_ORIGINAL_INFO_FIXED_V4.csv")

# ==================================================
# CHECK FILE
# ==================================================

print("File exists:", file.exists())
print("File path:", file)

if file.exists():
    size_gb = file.stat().st_size / (1024**3)
    print(f"File size: {size_gb:.2f} GB")
else:
    raise FileNotFoundError(f"Cannot find file: {file}")

# ==================================================
# DISPLAY SETTINGS
# ==================================================

pd.set_option("display.max_columns", None)
pd.set_option("display.max_rows", 200)
pd.set_option("display.width", 200)
pd.set_option("display.max_colwidth", 80)

# ==================================================
# MEMORY SAFE LOAD ONLY TOP 20 ROWS
# ==================================================

df_preview = pd.read_csv(
    file,
    nrows=20,          # only load first 20 rows
    low_memory=False,
    keep_default_na=False
)

# ==================================================
# SHOW BASIC INFO
# ==================================================

print("\nRows loaded:", len(df_preview))
print("Columns found:", len(df_preview.columns))

print("\n==============================")
print("COLUMN NAMES")
print("==============================")

for i, col in enumerate(df_preview.columns, start=1):
    print(f"{i}. {col}")

print("\n==============================")
print("COLUMN DTYPES FROM FIRST 20 ROWS")
print("==============================")

print(df_preview.dtypes)

print("\n==============================")
print("TOP 20 ROWS")
print("==============================")

display(df_preview)

print("\n==============================")
print("TOP 20 ROWS TRANSPOSED")
print("Better if too many columns")
print("==============================")

display(df_preview.T)

File exists: True
File path: FileVersion/IPEDS_ALL_ORIGINAL_INFO_FIXED_V4.csv
File size: 3.71 GB

Rows loaded: 20
Columns found: 242

COLUMN NAMES
1. unitid
2. cipcode
3. awlevel
4. crace15
5. crace16
6. part
7. crace01
8. crace02
9. crace03
10. crace04
11. crace05
12. crace06
13. crace07
14. crace08
15. crace09
16. crace10
17. crace11
18. crace12
19. crace13
20. crace14
21. bal_m
22. bal_w
23. imprac01
24. imprac02
25. imprac03
26. imprac04
27. imprac05
28. imprac06
29. imprac07
30. imprac08
31. imprac09
32. imprac10
33. imprac11
34. imprac12
35. imprac13
36. imprac14
37. imprac15
38. imprac16
39. ix01
40. ix02
41. ix03
42. ix04
43. ix05
44. ix06
45. ix07
46. ix08
47. ix09
48. ix10
49. ix11
50. ix12
51. ix13
52. ix14
53. ix15
54. ix16
55. cbalm
56. cbalw
57. xcrace01
58. xcrace02
59. xcrace03
60. xcrace04
61. xcrace05
62. xcrace06
63. xcrace07
64. xcrace08
65. xcrace09
66. xcrace10
67. xcrace11
68. xcrace12
69. xcrace13
70. xcrace14
71. xcrace15
72. xcrace16
73. xcrace17
74. crace17
7

,unitid,cipcode,awlevel,crace15,crace16,part,crace01,crace02,crace03,crace04,crace05,crace06,crace07,crace08,crace09,crace10,crace11,crace12,crace13,crace14,bal_m,bal_w,imprac01,imprac02,imprac03,imprac04,imprac05,imprac06,imprac07,imprac08,imprac09,imprac10,imprac11,imprac12,imprac13,imprac14,imprac15,imprac16,ix01,ix02,ix03,ix04,ix05,ix06,ix07,ix08,ix09,ix10,ix11,ix12,ix13,ix14,ix15,ix16,cbalm,cbalw,xcrace01,xcrace02,xcrace03,xcrace04,xcrace05,xcrace06,xcrace07,xcrace08,xcrace09,xcrace10,xcrace11,xcrace12,xcrace13,xcrace14,xcrace15,xcrace16,xcrace17,crace17,xcrace18,crace18,xcrace19,crace19,xcrace20,crace20,xcrace21,crace21,xcrace22,crace22,xcrace23,crace23,xcrace24,crace24,majornum,UNITID,CIPCODE,AWLEVEL,MAJORNUM,XCRACE01,CRACE01,XCRACE02,CRACE02,XCRACE03,CRACE03,XCRACE04,CRACE04,XCRACE05,CRACE05,XCRACE06,CRACE06,XCRACE07,CRACE07,XCRACE08,CRACE08,XCRACE09,CRACE09,XCRACE10,CRACE10,XCRACE11,CRACE11,XCRACE12,CRACE12,XCRACE13,CRACE13,XCRACE14,CRACE14,XCRACE15,CRACE15,XCRACE16,CRACE16,XCRACE17,CRACE17,XCRACE18,CRACE18,XCRACE19,CRACE19,XCRACE20,CRACE20,XCRACE21,CRACE21,XCRACE22,CRACE22,XCRACE23,CRACE23,XCRACE24,CRACE24,XCNRALM,CNRALM,XCNRALW,CNRALW,XCUNKNM,CUNKNM,XCUNKNW,CUNKNW,XCTOTALM,CTOTALM,XCTOTALW,CTOTALW,XCNRALT,CNRALT,XCUNKNT,CUNKNT,XCTOTALT,CTOTALT,XCHISPM,CHISPM,XCHISPW,CHISPW,XCAIANM,CAIANM,XCAIANW,CAIANW,XCASIAM,CASIAM,XCASIAW,CASIAW,XCBKAAM,CBKAAM,XCBKAAW,CBKAAW,XCNHPIM,CNHPIM,XCNHPIW,CNHPIW,XCWHITM,CWHITM,XCWHITW,CWHITW,XC2MORM,C2MORM,XC2MORW,C2MORW,XCHISPT,CHISPT,XCAIANT,CAIANT,XCASIAT,CASIAT,XCBKAAT,CBKAAT,XCNHPIT,CNHPIT,XCWHITT,CWHITT,XC2MORT,C2MORT,XDVCAIT,DVCAIT,XDVCAIM,DVCAIM,XDVCAIW,DVCAIW,XDVCAPT,DVCAPT,XDVCAPM,DVCAPM,XDVCAPW,DVCAPW,XDVCBKT,DVCBKT,XDVCBKM,DVCBKM,XDVCBKW,DVCBKW,XDVCHST,DVCHST,XDVCHSM,DVCHSM,XDVCHSW,DVCHSW,XDVCWHT,DVCWHT,XDVCWHM,DVCWHM,XDVCWHW,DVCWHW,CNRALW,CDISTEDP,year,award_level_code,award_level_name,degree_group,cip_code,degree_count,field_group,field_subgroup,major_name
0,100654,10102,5,11,0,,,,,,,,,,,,,,,,,,,,,,,,,,,,,,,,,,,,,,,,,,,,,,,,,,,,,,,,,,,,,,,,,,,,,,,,,,,,,,,,,,,,,,,,,,,,,,,,,,,,,,,,,,,,,,,,,,,,,,,,,,,,,,,,,,,,,,,,,,,,,,,,,,,,,,,,,,,,,,,,,,,,,,,,,,,,,,,,,,,,,,,,,,,,,,,,,,,,,,,,,,,,,,,,,,,,,,,,,,,,,,,,,,,,,1984,5,Bachelor,Bachelor,10102,0,COMMUNICATIONS TECHNOLOGIES/TECHNICIANS AND SUPPORT SERVICES,Unmapped / Nonstandard CIP Subgroup (10102),Unknown Major
1,100654,10102,7,14,0,,,,,,,,,,,,,,,,,,,,,,,,,,,,,,,,,,,,,,,,,,,,,,,,,,,,,,,,,,,,,,,,,,,,,,,,,,,,,,,,,,,,,,,,,,,,,,,,,,,,,,,,,,,,,,,,,,,,,,,,,,,,,,,,,,,,,,,,,,,,,,,,,,,,,,,,,,,,,,,,,,,,,,,,,,,,,,,,,,,,,,,,,,,,,,,,,,,,,,,,,,,,,,,,,,,,,,,,,,,,,,,,,,,,,1984,7,Master,Master,10102,0,COMMUNICATIONS TECHNOLOGIES/TECHNICIANS AND SUPPORT SERVICES,Unmapped / Nonstandard CIP Subgroup (10102),Unknown Major
2,100654,10103,5,1,1,,,,,,,,,,,,,,,,,,,,,,,,,,,,,,,,,,,,,,,,,,,,,,,,,,,,,,,,,,,,,,,,,,,,,,,,,,,,,,,,,,,,,,,,,,,,,,,,,,,,,,,,,,,,,,,,,,,,,,,,,,,,,,,,,,,,,,,,,,,,,,,,,,,,,,,,,,,,,,,,,,,,,,,,,,,,,,,,,,,,,,,,,,,,,,,,,,,,,,,,,,,,,,,,,,,,,,,,,,,,,,,,,,,,,1984,5,Bachelor,Bachelor,10103,0,COMMUNICATIONS TECHNOLOGIES/TECHNICIANS AND SUPPORT SERVICES,Unmapped / Nonstandard CIP Subgroup (10103),Unknown Major
3,100654,20201,5,3,0,,,,,,,,,,,,,,,,,,,,,,,,,,,,,,,,,,,,,,,,,,,,,,,,,,,,,,,,,,,,,,,,,,,,,,,,,,,,,,,,,,,,,,,,,,,,,,,,,,,,,,,,,,,,,,,,,,,,,,,,,,,,,,,,,,,,,,,,,,,,,,,,,,,,,,,,,,,,,,,,,,,,,,,,,,,,,,,,,,,,,,,,,,,,,,,,,,,,,,,,,,,,,,,,,,,,,,,,,,,,,,,,,,,,,1984,5,Bachelor,Bachelor,20201,0,Unmapped / Nonstandard CIP Field (20),Unmapped / Nonstandard CIP Subgroup (20201),Unknown Major
4,100654,20301,5,3,0,,,,,,,,,,,,,,,,,,,,,,,,,,,,,,,,,,,,,,,,,,,,,,,,,,,,,,,,,,,,,,,,,,,,,,,,,,,,,,,,,,,,,,,,,,,,,,,,,,,,,,,,,,,,,,,,,,,,,,,,,,,,,,,,,,,,,,,,,,,,,,,,,,,,,,,,,,,,,,,,,,,,,,,,,,,,,,,,,,,,,,,,,,,,,,,,,,,,,,,,,,,,,,,,,,,,,,,,,,,,,,,,,,,,,1984,5,Bachelor,Bachelor,20301,0,Unmapped / Nonstandard CIP Field (20),Unmapped / Nonstandard CIP Subgroup (20301),Unknown Major
5,100654,20399,7,11,3,,,,,,,,,,,,,,,,,,,,,,,,,,,,,,,,,,,,,,,,,,,,,,,,,,,,,,,,,,,,,,,,,,,,,,,,,,,,,,,


TOP 20 ROWS TRANSPOSED
Better if too many columns


,0,1,2,3,4,5,6,7,8,9,10,11,12,13,14,15,16,17,18,19
unitid,100654,100654,100654,100654,100654,100654,100654,100654,100654,100654,100654,100654,100654,100654,100654,100654,100654,100654,100654,100654
cipcode,10102,10102,10103,20201,20301,20399,20501,29999,29999,30499,40301,40301,60101,60201,60301,60401,60401,60501,60501,61401
awlevel,5,7,5,5,5,7,5,5,7,5,5,7,5,5,5,5,7,5,7,5
crace15,11,14,1,3,3,11,2,1,10,2,7,10,8,17,5,17,91,14,11,22
crace16,0,0,1,0,0,3,0,0,0,0,2,0,12,14,2,22,16,3,1,12
...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...
cip_code,10102,10102,10103,20201,20301,20399,20501,29999,29999,30499,40301,40301,60101,60201,60301,60401,60401,60501,60501,61401
degree_count,0,0,0,0,0,0,0,0,0,0,0,0,0,0,0,0,0,0,0,0
field_group,COMMUNICATIONS TECHNOLOGIES/TECHNICIANS AND SUPPORT SERVICES,COMMUNICATIONS TECHNOLOGIES/TECHNICIANS AND SUPPORT SERVICES,COMMUNICATIONS TECHNOLOGIES/TECHNICIANS AND SUPPORT SERVICES,Unmapped / Nonstandard CIP Field (20),Unmapped / Nonstandard CIP Field (20),Unmapped / Nonstandard CIP Field (20),Unmapped / Nonstandard CIP Field (20),MILITARY TECHNOLOGIES AND APPLIED SCIENCES,MILITARY TECHNOLOGIES AND APPLIED SCIENCES,MULTI/INTERDISCIPLINARY STUDIES,PHYSICAL SCIENCES,PHYSICAL SCIENCES,HEALTH PROFESSIONS RESIDENCY/FELLOWSHIP PROGRAMS,HEALTH PROFESSIONS RESIDENCY/FELLOWSHIP PROGRAMS,HEALTH PROFESSIONS RESIDENCY/FELLOWSHIP PROGRAMS,HEALTH PROFESSIONS RESIDENCY/FELLOWSHIP PROGRAMS,HEALTH PROFESSIONS RESIDENCY/FELLOWSHIP PROGRAMS,HEALTH PROFESSIONS RESIDENCY/FELLOWSHIP PROGRAMS,HEALTH PROFESSIONS RESIDENCY/FELLOWSHIP PROGRAMS,MEDICAL RESIDENCY/FELLOWSHIP PROGRAMS
field_subgroup,Unmapped / Nonstandard CIP Subgroup (10102),Unmapped / Nonstandard CIP Subgroup (10102),Unmapped / Nonstandard CIP Subgroup (10103),Unmapped / Nonstandard CIP Subgroup (20201),Unmapped / Nonstandard CIP Subgroup (20301),Unmapped / Nonstandard CIP Subgroup (20399),Unmapped / Nonstandard CIP Subgroup (20501),Unmapped / Nonstandard CIP Subgroup (29999),Unmapped / Nonstandard CIP Subgroup (29999),Unmapped / Nonstandard CIP Subgroup (30499),Unmapped / Nonstandard CIP Subgroup (40301),Unmapped / Nonstandard CIP Subgroup (40301),Unmapped / Nonstandard CIP Subgroup (60101),Unmapped / Nonstandard CIP Subgroup (60201),Unmapped / Nonstandard CIP Subgroup (60301),Unmapped / Nonstandard CIP Subgroup (60401),Unmapped / Nonstandard CIP Subgroup (60401),Unmapped / Nonstandard CIP Subgroup (60501),Unmapped / Nonstandard CIP Subgroup (60501),Unmapped / Nonstandard CIP Subgroup (61401)


# Redo

In [ ]:
import pandas as pd
from pathlib import Path

# ==================================================
# FILE
# ==================================================

file = Path("FileVersion/IPEDS_ALL_ORIGINAL_INFO_FIXED_V4.csv")

# ==================================================
# MEMORY SAFE SETTINGS
# ==================================================

chunksize = 200_000

bad_rows = []
total = 0
bad_count = 0

# ==================================================
# CHECK UNKNOWN MAJOR ROWS
# ==================================================

for chunk_num, chunk in enumerate(pd.read_csv(
    file,
    chunksize=chunksize,
    low_memory=False,
    keep_default_na=False
), start=1):

    total += len(chunk)

    major_col = chunk["major_name"].astype(str).str.strip()

    bad = chunk[
        major_col.str.contains("Unknown Major", case=False, na=False)
        | major_col.eq("")
        | major_col.eq("Unknown")
    ].copy()

    bad_count += len(bad)

    if len(bad) > 0:
        bad_rows.append(bad.head(50))

    print(
        f"\rChecked chunk {chunk_num:,} | rows checked: {total:,} | bad major rows found: {bad_count:,}",
        end=""
    )

print("\n\nDONE")
print(f"Total rows checked: {total:,}")
print(f"Bad / unknown major rows: {bad_count:,}")

# ==================================================
# SHOW BAD ROWS
# ==================================================

if bad_rows:
    bad_preview = pd.concat(bad_rows, ignore_index=True)

    print("\nColumns:")
    print(list(bad_preview.columns))

    print("\nBad major preview:")
    display(bad_preview.head(100))

    print("\nBad major rows transposed:")
    display(bad_preview.head(20).T)
else:
    print("No Unknown Major rows found.")

Checked chunk 24 | rows checked: 4,800,000 | bad major rows found: 2,979,633

In [ ]:
import pandas as pd
from pathlib import Path

file = Path("FileVersion/IPEDS_ALL_ORIGINAL_INFO_FIXED_V4.csv")

chunksize = 200_000
bad_parts = []

for chunk in pd.read_csv(
    file,
    chunksize=chunksize,
    low_memory=False,
    keep_default_na=False
):
    bad = chunk[
        chunk["major_name"].astype(str).str.contains("Unknown Major", case=False, na=False)
    ].copy()

    if len(bad) > 0:
        bad_parts.append(bad)

if bad_parts:
    bad_all = pd.concat(bad_parts, ignore_index=True)

    keep_cols = [
        col for col in [
            "year",
            "unitid",
            "institution_name",
            "cipcode",
            "CIPCODE",
            "cip_code",
            "major_name",
            "field_group",
            "field_subgroup",
            "clean_major_name",
            "program_title",
            "CIPDESC",
            "cipdesc",
            "degree_count"
        ]
        if col in bad_all.columns
    ]

    display(bad_all[keep_cols].head(100))

    print("Unique bad major rows:")
    display(
        bad_all[keep_cols]
        .drop_duplicates()
        .head(100)
    )

else:
    print("No Unknown Major rows found.")